# EduGuard AI: Student Performance Prediction Model
### SDG 4: Quality Education | IBM SkillsBuild ML & Applied AI Internship

**Goal:** Train a machine learning model that predicts a student's **exam score** from study habits and
life factors (hours studied, attendance, sleep, motivation, etc.). The app uses this prediction to flag
**at-risk students early**, so teachers and the AI Study Coach can step in *before* the exam.

**How to use this notebook:** Run each cell **from top to bottom** by clicking it and pressing `Shift + Enter`.
Read the text cells and the `#` comments in the code. They explain what every line does.

**What you will get at the end:**
- `model.pkl`: the trained model (the "brain" of the app)
- `model_info.json`: model accuracy and settings the app needs
- `requirements.txt`: the list of libraries the app needs
- 5 chart images (`.png`) to use in your PPT and concept note

## Step 1: Import the libraries
A **library** is ready-made code that someone else wrote. We import it instead of writing everything ourselves.
- `pandas` → works with tables of data (like Excel in Python)
- `numpy` → maths on lists of numbers
- `matplotlib` → draws charts
- `sklearn` (scikit-learn) → the machine learning toolkit
- `joblib` → saves the trained model to a file

In [ ]:
import pandas as pd                    # pandas = tables of data; "pd" is a short nickname
import numpy as np                     # numpy = fast maths on numbers
import matplotlib.pyplot as plt        # matplotlib = charts
import json, os, glob                  # built-in Python tools: json files, folders, file search
import joblib                          # saves / loads the trained model
import sklearn                         # the machine learning library (we import parts of it below)

# --- pieces of scikit-learn we will use ---
from sklearn.model_selection import train_test_split        # splits data into training and testing parts
from sklearn.compose import ColumnTransformer               # applies different preparation to different columns
from sklearn.preprocessing import OneHotEncoder             # turns text categories (Low/Medium/High) into numbers
from sklearn.pipeline import Pipeline                       # chains "prepare data" + "model" into one object
from sklearn.linear_model import LinearRegression           # Model 1: draws the best straight-line relationship
from sklearn.ensemble import RandomForestRegressor          # Model 2: many decision trees voting together
from sklearn.ensemble import GradientBoostingRegressor      # Model 3: trees that learn from each other's mistakes
from sklearn.metrics import r2_score, mean_absolute_error   # ways to measure how good a model is
from sklearn.inspection import permutation_importance       # finds which inputs matter most

# Chart style used for every chart (consistent look for your PPT)
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#52514e"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#bbbbbb", "axes.labelcolor": GREY, "xtick.color": GREY,
                     "ytick.color": GREY, "axes.titleweight": "bold", "axes.titlesize": 13})

print("All libraries loaded. scikit-learn version:", sklearn.__version__)

## Step 2: Load the dataset
We use the **Student Performance Factors** dataset from Kaggle (about 6,600 students, 20 columns).

The cell below tries 3 ways, in order:
1. If you already uploaded `StudentPerformanceFactors.csv` to Colab, it uses that file.
2. Otherwise it downloads the file from Kaggle automatically with `kagglehub`.
3. If that fails, a **"Choose Files"** button appears. Upload the CSV you downloaded from
   https://www.kaggle.com/datasets/lainguyn123/student-performance-factors

In [ ]:
CSV_NAME = "StudentPerformanceFactors.csv"     # the file name of the dataset

if os.path.exists(CSV_NAME):                   # Way 1: file is already here
    csv_path = CSV_NAME
else:
    try:                                       # Way 2: download from Kaggle automatically
        import kagglehub
        folder = kagglehub.dataset_download("lainguyn123/student-performance-factors")
        csv_path = glob.glob(os.path.join(folder, "*.csv"))[0]     # find the .csv inside the downloaded folder
    except Exception as error:                 # Way 3: upload by hand
        print("Automatic download did not work:", error)
        print("Please upload StudentPerformanceFactors.csv using the button below.")
        from google.colab import files
        uploaded = files.upload()              # opens a 'Choose Files' button
        csv_path = list(uploaded.keys())[0]

df = pd.read_csv(csv_path)                     # read the CSV file into a table called df ("data frame")
df.to_csv(CSV_NAME, index=False)               # keep a copy named StudentPerformanceFactors.csv (the app needs it)
print("Loaded", df.shape[0], "students and", df.shape[1], "columns")
df.head()                                      # show the first 5 rows

## Step 3: Explore the data (EDA = Exploratory Data Analysis)
Before training a model, a data scientist always *looks* at the data first:
what columns exist, what type they are, and whether anything is missing.

In [ ]:
df.info()        # column names, data types (int = whole numbers, object/str = text) and non-missing counts

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum()[df.isnull().sum() > 0])    # only show columns that actually have missing values

In [ ]:
df.describe().round(1)     # summary numbers (count, mean, min, max...) for every numeric column

### Charts that tell the story
These charts are saved as `.png` files. **Put them in your PPT and concept note.**

In [ ]:
# Chart 1: How are exam scores spread out?
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["Exam_Score"], bins=30, color=BLUE, edgecolor="white", linewidth=2)   # histogram = count of students per score range
ax.set_title("How exam scores are distributed")
ax.set_xlabel("Exam score"); ax.set_ylabel("Number of students")
ax.grid(axis="y", color="#e8e8e8"); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig("chart1_score_distribution.png"); plt.show()

In [ ]:
# Chart 2: Do attendance and study hours affect the score? (average score per group)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

att_groups = pd.cut(df["Attendance"], bins=[59, 70, 80, 90, 100], labels=["60-70%", "71-80%", "81-90%", "91-100%"])
att_avg = df.groupby(att_groups, observed=True)["Exam_Score"].mean()      # average score in each attendance group
axes[0].plot(att_avg.index.astype(str), att_avg.values, color=BLUE, linewidth=2, marker="o", markersize=9)   # dots joined by a line
axes[0].set_title("Higher attendance → higher scores"); axes[0].set_xlabel("Attendance"); axes[0].set_ylabel("Average exam score")
axes[0].set_ylim(60, 74); axes[0].grid(axis="y", color="#e8e8e8")
for i, v in enumerate(att_avg.values): axes[0].text(i, v + 0.4, f"{v:.1f}", ha="center", color=GREY)

hrs_groups = pd.cut(df["Hours_Studied"], bins=[0, 10, 20, 30, 50], labels=["0-10", "11-20", "21-30", "31+"])
hrs_avg = df.groupby(hrs_groups, observed=True)["Exam_Score"].mean()
axes[1].plot(hrs_avg.index.astype(str), hrs_avg.values, color=BLUE, linewidth=2, marker="o", markersize=9)
axes[1].set_title("More study hours → higher scores"); axes[1].set_xlabel("Hours studied per week")
axes[1].set_ylim(60, 74); axes[1].grid(axis="y", color="#e8e8e8")
for i, v in enumerate(hrs_avg.values): axes[1].text(i, v + 0.4, f"{v:.1f}", ha="center", color=GREY)

plt.tight_layout(); plt.savefig("chart2_attendance_hours.png"); plt.show()

## Step 4: Clean the data
Real data is messy. Two fixes:
1. **Missing values:** 3 text columns have a few blanks. We fill each blank with the **most common value** (the "mode").
2. **Impossible values:** one exam score is above 100. We cap scores at 100.

In [ ]:
for col in ["Teacher_Quality", "Parental_Education_Level", "Distance_from_Home"]:
    most_common = df[col].mode()[0]              # mode = the value that appears most often
    df[col] = df[col].fillna(most_common)        # fill the blanks with it

df["Exam_Score"] = df["Exam_Score"].clip(upper=100)    # any score above 100 becomes 100
print("Missing values left:", int(df.isnull().sum().sum()))

## Step 5: Separate inputs (X) and the answer we want to predict (y)
- **X (features)** = everything we know about the student (19 columns)
- **y (target)** = `Exam_Score`, which the model must learn to predict

Then we **split** the students: **80% for training** (the model learns from these) and **20% for testing**
(the model has *never seen* these, so they show how well it works on new students).

In [ ]:
TARGET = "Exam_Score"
X = df.drop(columns=[TARGET])      # inputs: all columns except Exam_Score
y = df[TARGET]                     # output: Exam_Score

text_cols = X.select_dtypes(exclude="number").columns.tolist()   # columns with text like 'Low', 'Yes', 'Public'
number_cols = X.select_dtypes(include="number").columns.tolist() # columns with numbers like hours, attendance
print("Text columns  :", text_cols)
print("Number columns:", number_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,           # 20% of students kept aside for testing
    random_state=42)               # fixed 'seed' so you get the same split every time you run it
print("Training students:", len(X_train), "| Testing students:", len(X_test))

## Step 6: Prepare the text columns
ML models only understand numbers. **One-hot encoding** turns a text column into yes/no number columns. Example:

| Motivation_Level | → | Motivation_Low | Motivation_Medium | Motivation_High |
|---|---|---|---|---|
| Medium | → | 0 | 1 | 0 |

We put this step and the model together in a **Pipeline**. The app can then give the raw student details
to one object, and it handles the preparation and the prediction.

In [ ]:
def make_pipeline(model):
    prep = ColumnTransformer(
        transformers=[("text", OneHotEncoder(handle_unknown="ignore"), text_cols)],  # one-hot encode text columns
        remainder="passthrough")                                                     # number columns pass through unchanged
    return Pipeline([("prep", prep), ("model", model)])    # step 1: prepare, step 2: model

## Step 7: Train 3 different models and compare them
We don't guess which algorithm is best. We **try three and measure** each one:
- **R² score** (0 to 1): how much of the variation in scores the model explains. **Closer to 1 = better.**
- **MAE (Mean Absolute Error):** on average, how many marks the prediction is off by. **Lower = better.**

In [ ]:
candidates = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),   # 300 trees
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

results = []                 # we will store each model's scores here
trained = {}                 # and the trained models here
for name, model in candidates.items():
    pipe = make_pipeline(model)
    pipe.fit(X_train, y_train)                       # TRAINING: the model learns patterns from the training students
    preds = pipe.predict(X_test)                     # TESTING: predict scores for students it has never seen
    r2 = r2_score(y_test, preds)                     # compare predictions with the real scores
    mae = mean_absolute_error(y_test, preds)
    results.append({"Model": name, "R2 score": round(r2, 3), "MAE (marks)": round(mae, 2)})
    trained[name] = pipe
    print(f"{name:18s} → R² = {r2:.3f}, average error = {mae:.2f} marks")

results_df = pd.DataFrame(results).sort_values("R2 score", ascending=False)
results_df

In [ ]:
# Chart 3: model comparison (good slide for your PPT)
fig, ax = plt.subplots(figsize=(8, 3.6))
order = results_df.sort_values("R2 score")
colors = [BLUE if i == len(order) - 1 else "#a9c7ee" for i in range(len(order))]   # highlight the winner
ax.barh(order["Model"], order["R2 score"], color=colors, height=0.55)
for i, v in enumerate(order["R2 score"]): ax.text(v + 0.01, i, f"{v:.3f}", va="center", color=GREY)
ax.set_xlim(0, 1); ax.set_xlabel("R² score (higher is better)")
ax.set_title("Comparing 3 machine learning models")
plt.tight_layout(); plt.savefig("chart3_model_comparison.png"); plt.show()

## Step 8: Pick the best model and check it visually
If the predictions are good, the dots in the chart below sit close to the diagonal line
(diagonal = "predicted exactly right").

> **Interesting finding for your presentation:** a simple Linear Regression usually beats the complex
> tree models here. That means the factors add up in a fairly straight-line way. *Simpler is not always worse.*

In [ ]:
best_name = results_df.iloc[0]["Model"]          # model with the highest R² (top row of the sorted table)
best_model = trained[best_name]
best_preds = best_model.predict(X_test)
print("Best model:", best_name)

# Chart 4: predicted vs actual
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_test, best_preds, s=12, alpha=0.45, color=BLUE, edgecolors="none")
lims = [55, 100]
ax.plot(lims, lims, color=ORANGE, linewidth=2, label="Perfect prediction")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual exam score"); ax.set_ylabel("Predicted exam score")
ax.set_title(f"{best_name}: predicted vs actual"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig("chart4_predicted_vs_actual.png"); plt.show()

## Step 9: Which factors matter most?
**Permutation importance:** we shuffle one column at a time and see how much worse the model gets.
If shuffling "Attendance" makes predictions much worse, attendance is very important.

In [ ]:
imp = permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
importance = pd.Series(imp.importances_mean, index=X_test.columns).sort_values()   # one number per input column
top = importance.tail(10)                           # the 10 most important factors

# Chart 5: top factors
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(top.index.str.replace("_", " "), top.values, color=BLUE, height=0.6)
ax.set_xlabel("Importance (drop in accuracy when this factor is shuffled)")
ax.set_title("Top 10 factors that affect exam scores")
plt.tight_layout(); plt.savefig("chart5_feature_importance.png"); plt.show()

## Step 10: Try the model on one new student
Change the numbers and words below and run the cell again to see how the prediction changes.

In [ ]:
new_student = pd.DataFrame([{
    "Hours_Studied": 12, "Attendance": 68, "Parental_Involvement": "Low", "Access_to_Resources": "Medium",
    "Extracurricular_Activities": "No", "Sleep_Hours": 6, "Previous_Scores": 62, "Motivation_Level": "Low",
    "Internet_Access": "Yes", "Tutoring_Sessions": 0, "Family_Income": "Low", "Teacher_Quality": "Medium",
    "School_Type": "Public", "Peer_Influence": "Negative", "Physical_Activity": 2, "Learning_Disabilities": "No",
    "Parental_Education_Level": "High School", "Distance_from_Home": "Far", "Gender": "Female",
}])
predicted = best_model.predict(new_student)[0]
print(f"Predicted exam score: {predicted:.1f}")

# Risk levels are based on how all students scored:
at_risk_cutoff = float(y.quantile(0.25))     # bottom 25% of students
watch_cutoff = float(y.quantile(0.50))       # below the middle (median) student
if predicted <= at_risk_cutoff:   print("Risk level: HIGH RISK (bottom 25% of students)")
elif predicted <= watch_cutoff:   print("Risk level: NEEDS ATTENTION (below average)")
else:                             print("Risk level: ON TRACK")

## Step 11: Save everything the app needs

In [ ]:
joblib.dump(best_model, "model.pkl")         # save the trained model (pipeline) to a file

info = {                                     # extra information the app shows on its "Model Insights" page
    "best_model": best_name,
    "r2": float(results_df.iloc[0]["R2 score"]),
    "mae": float(results_df.iloc[0]["MAE (marks)"]),
    "all_results": results,
    "n_students": int(len(df)),
    "at_risk_cutoff": at_risk_cutoff,
    "watch_cutoff": watch_cutoff,
    "feature_importance": {k: round(float(v), 4) for k, v in importance.sort_values(ascending=False).items()},
    "sklearn_version": sklearn.__version__,
}
with open("model_info.json", "w") as f:
    json.dump(info, f, indent=2)

# requirements.txt tells Streamlit Cloud which libraries to install.
# We pin scikit-learn to the SAME version used here, so model.pkl loads correctly in the app.
with open("requirements.txt", "w") as f:
    f.write(f"streamlit\npandas\nnumpy\nmatplotlib\njoblib\ngoogle-genai\nscikit-learn=={sklearn.__version__}\n")

print("Saved: model.pkl, model_info.json, requirements.txt, StudentPerformanceFactors.csv and 5 chart PNGs")

In [ ]:
# Download all the files to your computer (Colab only; this cell is skipped elsewhere)
try:
    from google.colab import files
    for name in ["model.pkl", "model_info.json", "requirements.txt", "StudentPerformanceFactors.csv",
                 "chart1_score_distribution.png", "chart2_attendance_hours.png", "chart3_model_comparison.png",
                 "chart4_predicted_vs_actual.png", "chart5_feature_importance.png"]:
        files.download(name)
except ImportError:
    print("Not running in Colab; files are saved in the current folder.")

## Done!
Upload `model.pkl`, `model_info.json`, `requirements.txt` and `StudentPerformanceFactors.csv`
to your GitHub repository, next to `app.py`. Then follow the deployment steps in the guide.

**Note down these numbers for your PPT:** the best model's name, its R² score and its MAE.